# Practice Lab: Agentic RAG (Lesson 8.10)

Module 8 · 8 exercises · Routing + grading + CRAG + Adaptive + circuit breakers

Exercises 1, 2, 3, 6, 8 run locally (pure Python).


# Lesson 8.10: Agentic RAG — RAG That Thinks

8 exercises: 2E + 3M + 3C

Exercises 1, 2, 3, 6, 8 run **locally** (pure Python). Ex 4, 5, 7 are architecture/design.


In [ ]:
import numpy as np
import hashlib, time


---
## Exercise 1 (Easy): Query Router


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
def classify(q):
    q = q.lower()
    if any(k in q for k in ["hello","hi ","hey","thank","2+2","calculate","how are you","good morning"]): return "DIRECT"
    if any(k in q for k in ["latest","yesterday","today","current","news","weather","who won","stock","python version"]): return "WEB"
    return "DOCS"

tests = [("What is 2+2?","DIRECT"),("Hello!","DIRECT"),("Thanks!","DIRECT"),
         ("Refund policy?","DOCS"),("Course cost?","DOCS"),("Prerequisites?","DOCS"),
         ("Live classes?","DOCS"),("EMI options?","DOCS"),
         ("IPL match yesterday?","WEB"),("Latest Python version?","WEB"),
         ("Infosys stock price?","WEB"),("Weather today?","WEB")]

print("Query Router:")
ok = 0; counts = {"DIRECT":0,"DOCS":0,"WEB":0}
for q, exp in tests:
    pred = classify(q); match = pred == exp
    if match: ok += 1
    counts[pred] += 1
    print(f"  [{'OK' if match else '!!'}] [{pred:<6}] {q[:35]}...")

print(f"\nAccuracy: {ok}/{len(tests)} ({ok/len(tests):.0%})")
d_pct = counts["DIRECT"]/len(tests)
print(f"DIRECT={counts['DIRECT']} ({d_pct:.0%} skip retrieval)")
print(f"Production: 30-50% retrieval calls saved")


</details>


---
## Exercise 2 (Easy): Retrieval Grader


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import hashlib
import numpy as np

def fe(t, dim=64):
    h = hashlib.sha256(t.lower().encode()).digest()
    v = np.array([b/255.0 for b in h[:dim]])
    for w, i in {"refund":0,"price":2,"cost":3,"course":4,"prereq":5,"emi":7,"blockchain":10}.items():
        if w in t.lower() and i < dim: v[i] += 0.5
    return v / np.linalg.norm(v)

docs = ["Refund: full within 7 days", "GenAI 14999 rupees lifetime", "Prerequisites: Python math",
        "EMI via Razorpay 2500/month", "Live classes 7 PM IST"]
dembs = [fe(d) for d in docs]

def retrieve(q, k=3):
    qe = fe(q)
    scores = sorted([(i, np.dot(qe,dembs[i])/(np.linalg.norm(qe)*np.linalg.norm(dembs[i]))) for i in range(len(docs))], key=lambda x:-x[1])[:k]
    return [(docs[i], round(s,3)) for i,s in scores]

def grade(q, doc, th=0.15):
    stops = {"what","is","the","a","how","do","does","can","i","for","in","to","of","and"}
    qw = set(q.lower().split()) - stops; dw = set(doc.lower().split())
    ov = len(qw & dw)/max(len(qw),1)
    return {"relevant": ov >= th, "score": round(ov, 2)}

for q in ["Refund policy?","Course cost?","Teach blockchain?","Pay EMI?","Prerequisites?"]:
    retrieved = retrieve(q)
    print(f"  Q: {q}")
    for doc, sim in retrieved:
        g = grade(q, doc)
        st = "RELEVANT" if g["relevant"] else "FILTERED"
        print(f"    [{st}] ({g['score']:.2f}) {doc[:35]}...")
    print()

print(f"Without grading: all docs feed LLM (noise)")
print(f"With grading: only relevant -> cleaner answers")


</details>


---
## Exercise 3 (Medium): CRAG with Web Fallback


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import hashlib
import numpy as np

def fe(t, dim=64):
    h = hashlib.sha256(t.lower().encode()).digest()
    v = np.array([b/255.0 for b in h[:dim]])
    for w, i in {"refund":0,"price":2,"course":4,"emi":7,"blockchain":10,"gpt5":11}.items():
        if w in t.lower() and i < dim: v[i] += 0.5
    return v / np.linalg.norm(v)

docs = ["Refund: full 7 days 50% 8-30", "GenAI 14999 rupees lifetime", "Prerequisites: Python math",
        "EMI Razorpay 2500/month", "Live classes 7 PM IST"]
dembs = [fe(d) for d in docs]

def retrieve(q, k=2):
    qe = fe(q)
    scores = sorted([(i, np.dot(qe,dembs[i])/(np.linalg.norm(qe)*np.linalg.norm(dembs[i]))) for i in range(len(docs))], key=lambda x:-x[1])[:k]
    return [(docs[i], round(s,3)) for i,s in scores]

def crag_eval(q, retrieved):
    stops = {"what","is","the","a","how","do","does","can","i","for","in","to","of","and"}
    qw = set(q.lower().split()) - stops
    scores = [len(qw & set(d.lower().split()))/max(len(qw),1) for d, _ in retrieved]
    avg = sum(scores)/max(len(scores),1)
    if avg >= 0.3: return "CORRECT", avg
    elif avg <= 0.05: return "INCORRECT", avg
    return "AMBIGUOUS", avg

print("CRAG (Corrective RAG):")
counts = {"CORRECT":0,"INCORRECT":0,"AMBIGUOUS":0}
for q in ["Refund policy?","Course cost?","EMI options?","Blockchain courses?","GPT-5 date?","Weather today?"]:
    ret = retrieve(q)
    action, conf = crag_eval(q, ret)
    counts[action] += 1
    strategy = {"CORRECT":"Refine docs","INCORRECT":"Web search","AMBIGUOUS":"Docs + web"}[action]
    print(f"  [{action:<10}] ({conf:.2f}) {q:<28} -> {strategy}")

print(f"\nDistribution: {counts}")
print(f"Hallucination: 10.5% (vs ~25% naive). Overhead: ~240ms")
print(f"Plug-and-play over ANY existing RAG")


</details>


---
## Exercise 6 (Challenge): Adaptive RAG


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class AdaptiveRAG:
    def classify(self, q):
        q = q.lower()
        if any(k in q for k in ["hello","hi ","2+2","thank","calculate"]): return "simple"
        if any(k in q for k in ["compare","relationship","across all","what connects","step by step"]): return "complex"
        return "moderate"
    def process(self, q):
        c = self.classify(q)
        costs = {"simple":0.001,"moderate":0.005,"complex":0.015}
        calls = {"simple":0,"moderate":1,"complex":3}
        return {"complexity":c,"calls":calls[c],"cost":costs[c]}

ar = AdaptiveRAG()
queries = [("Hello!","simple"),("What is 2+2?","simple"),("Thanks!","simple"),
           ("Refund policy?","moderate"),("Course cost?","moderate"),("Live classes?","moderate"),
           ("Compare all courses","complex"),("What connects prereqs to placement?","complex"),
           ("Step by step explain RAG","complex")]

print("Adaptive RAG:")
tc_adapt = tc_always = 0; ok = 0
for q, exp in queries:
    r = ar.process(q); match = r["complexity"] == exp
    if match: ok += 1
    tc_adapt += r["cost"]; tc_always += 0.005
    print(f"  [{r['complexity']:<8}] {q[:35]}... calls={r['calls']} ${r['cost']:.3f}")

print(f"\nAccuracy: {ok}/{len(queries)}")
print(f"Always: ${tc_always:.3f} | Adaptive: ${tc_adapt:.3f} | Savings: {(1-tc_adapt/tc_always):.0%}")
print(f"\nLadder: CRAG(plug-play) -> Adaptive(cost) -> Self-RAG(5.8% halluc)")


</details>


---
## Exercise 8 (Challenge): Production Circuit Breaker


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class CB:
    def __init__(self, ft=2, dt=2):
        self.state = "CLOSED"; self.fails = 0; self.deg = 0; self.ft = ft; self.dt = dt; self.hist = []
    def _trans(self, new, reason):
        self.hist.append(f"{self.state}->{new}: {reason}"); self.state = new
    def call(self, q, ag_fn, simple_fn, cache_fn, static_fn):
        if self.state == "OPEN": return static_fn(q), "STATIC"
        try:
            r = ag_fn(q)
            if "[HALLUC]" in r:
                self.deg += 1
                if self.state == "CLOSED": self._trans("DEGRADED","Semantic fail")
                if self.deg >= self.dt: self._trans("OPEN","Too many semantic fails"); return cache_fn(q), "CACHED"
                return simple_fn(q), "SIMPLE"
            if self.state == "DEGRADED": self._trans("CLOSED","Quality restored")
            self.fails = 0; self.deg = 0; return r, "AGENTIC"
        except:
            self.fails += 1
            if self.fails >= self.ft: self._trans("OPEN","Errors"); return static_fn(q), "STATIC"
            return simple_fn(q), "SIMPLE"

def ag(q):
    if "crash" in q.lower(): raise Exception("fail")
    if "halluc" in q.lower(): return "[HALLUC] bad answer"
    return f"Agentic: {q[:25]}..."
def si(q): return f"Simple: {q[:25]}..."
def ca(q): return f"Cached: generic"
def st(q): return f"Static: contact support"

cb = CB()
for q in ["Refund policy?","Course cost?","halluc topic","another halluc","prerequisites?","more queries"]:
    r, level = cb.call(q, ag, si, ca, st)
    print(f"  [{cb.state:<8}] [{level:<8}] {q[:25]}... -> {r[:30]}...")

print(f"\nTransitions: {cb.hist}")
print(f"Fallback: Agentic -> Simple -> Cached -> Static")
print(f"DEGRADED: LLMs return 200 while hallucinating!")


</details>


---
## Exercise 4 (Medium): LangGraph CRAG Topology
Architecture/design. See practice-lab-8_10.html.


In [ ]:
# Exercise 4: LangGraph CRAG Topology
pass


---
## Exercise 5 (Medium): LlamaIndex RouterQueryEngine
Architecture/design. See practice-lab-8_10.html.


In [ ]:
# Exercise 5: LlamaIndex RouterQueryEngine
pass


---
## Exercise 7 (Challenge): Model Tiering + Caching
Architecture/design. See practice-lab-8_10.html.


In [ ]:
# Exercise 7: Model Tiering + Caching
pass
